# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [2]:

from pathlib import Path
import polars as pl
import duckdb

# Lokasi utama proyek
PROJECT_ROOT = Path("/home/jovyan/work")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Pastikan folder output tersedia
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Cari semua file Parquet, termasuk di subfolder
PARQUET_FILES = sorted(RAW_DIR.rglob("*.parquet"))

if not PARQUET_FILES:
    raise FileNotFoundError(
        f"Tidak ada file Parquet ditemukan di {RAW_DIR}. "
        "Pastikan proses download dataset sudah selesai."
    )

print(f"Jumlah file Parquet ditemukan: {len(PARQUET_FILES)}")
for file in PARQUET_FILES:
    print(f"- {file}")


Jumlah file Parquet ditemukan: 6
- /home/jovyan/work/data/raw/data/train-00000-of-00006-042e4a2154d9270b.parquet
- /home/jovyan/work/data/raw/data/train-00001-of-00006-8666365fa80b1ec5.parquet
- /home/jovyan/work/data/raw/data/train-00002-of-00006-8ac9b553e82a7f8f.parquet
- /home/jovyan/work/data/raw/data/train-00003-of-00006-25183230856f688a.parquet
- /home/jovyan/work/data/raw/data/train-00004-of-00006-a7a9cb4b60d46e79.parquet
- /home/jovyan/work/data/raw/data/train-00005-of-00006-a7fd9cf190643493.parquet


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [4]:

import duckdb

connection = duckdb.connect()

# Gunakan file Parquet asli sebagai sumber data DuckDB
file_paths = [str(file) for file in PARQUET_FILES]

# Profiling statistik kolom
file_list_sql = "[" + ", ".join(
    "'" + path.replace("\\", "/").replace("'", "''") + "'"
    for path in file_paths
) + "]"

query = f"""
SUMMARIZE
SELECT *
FROM read_parquet({file_list_sql})
"""

profil = connection.execute(query).pl()

display(profil)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""qid""","""BIGINT""","""1""","""74661602""",264619,"""2798925.6786795994""","""9036720.75363977""","""26963""","""105209""","""455760""",1050257,0.00
"""question""","""VARCHAR""","""!!! I found an interesting sol…","""﻿ What would a political syste…",374355,null,null,null,null,null,1050257,0.00
"""date""","""VARCHAR""","""2008/08/01""","""2023/03/04""",4030,null,null,null,null,null,1050257,0.00
"""metadata""","""VARCHAR[]""","""['https://3dprinting.meta.stac…","""['https://writers.stackexchang…",374242,null,null,null,null,null,1050257,0.00
"""response_j""","""VARCHAR""","""""","""ﷲ = الله ====""",595801,null,null,null,null,null,1050257,0.00
"""response_k""","""VARCHAR""","""! Just as well that I have a…","""Watch only produces notificat…",631183,null,null,null,null,null,1050257,0.00


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [6]:

from pathlib import Path
import polars as pl

# Lokasi dataset
RAW_DIR = Path("/home/jovyan/work/data/raw")

# Cari semua file Parquet
PARQUET_FILES = sorted(RAW_DIR.rglob("*.parquet"))

if not PARQUET_FILES:
    raise FileNotFoundError(
        f"File Parquet tidak ditemukan di {RAW_DIR}. "
        "Periksa apakah dataset sudah selesai diunduh."
    )

print(f"File Parquet ditemukan: {len(PARQUET_FILES)}")

# Buat ulang variabel df
df = pl.scan_parquet([str(file) for file in PARQUET_FILES])

# Hitung missing values
columns = df.collect_schema().names()
total_rows = df.select(pl.len()).collect().item()

missing_values = df.select([
    pl.col(column).null_count().alias(column)
    for column in columns
]).collect()

missing_report = pl.DataFrame({
    "nama_kolom": columns,
    "jumlah_null": [
        missing_values[column][0] for column in columns
    ],
}).with_columns(
    (
        pl.col("jumlah_null") / total_rows * 100
    ).round(2).alias("persentase_null")
).sort("persentase_null", descending=True)

display(missing_report)
print(f"Total baris: {total_rows:,}")


File Parquet ditemukan: 6


nama_kolom,jumlah_null,persentase_null
str,i64,f64
"""qid""",0,0.0
"""question""",0,0.0
"""date""",0,0.0
"""metadata""",0,0.0
"""response_j""",0,0.0
"""response_k""",0,0.0


Total baris: 1,050,257


## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [7]:

import polars as pl

# Pastikan df sudah dibuat menggunakan pl.scan_parquet(...)
schema = df.collect_schema()
columns = schema.names()

# Hitung jumlah baris
total_rows = df.select(pl.len()).collect().item()

# Identifikasi kolom berdasarkan tipe data aktual
numeric_columns = [
    name for name, dtype in schema.items()
    if dtype.is_numeric()
]

text_columns = [
    name for name, dtype in schema.items()
    if dtype == pl.String
]

print("Kolom numerik:", numeric_columns)
print("Kolom teks:", text_columns)

# Buat strategi penanganan missing values
cleaning_expressions = []

for column in numeric_columns:
    cleaning_expressions.append(
        pl.col(column).fill_null(
            df.select(pl.col(column).median()).collect().item()
        ).alias(column)
    )

for column in text_columns:
    cleaning_expressions.append(
        pl.col(column)
        .fill_null("Tidak tersedia")
        .alias(column)
    )

# Terapkan hanya pada kolom yang teridentifikasi
df_bersih = df.with_columns(cleaning_expressions)

print("Penanganan missing values selesai.")


Kolom numerik: ['qid']
Kolom teks: ['question', 'date', 'response_j', 'response_k']
Penanganan missing values selesai.


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [8]:

import polars as pl

# Pastikan df_bersih tersedia dari tahap sebelumnya
schema = df_bersih.collect_schema()

# Kandidat kolom numerik
numeric_columns = [
    name
    for name, dtype in schema.items()
    if dtype.is_numeric() and name.lower() not in ["qid", "id"]
]

print("Kolom numerik yang diperiksa:", numeric_columns)

outlier_results = []

for column in numeric_columns:
    stats = df_bersih.select([
        pl.col(column).quantile(0.25).alias("q1"),
        pl.col(column).quantile(0.75).alias("q3"),
    ]).collect()

    q1 = stats["q1"][0]
    q3 = stats["q3"][0]

    if q1 is None or q3 is None:
        continue

    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    result = df_bersih.select(
        (
            (pl.col(column) < lower_bound) |
            (pl.col(column) > upper_bound)
        ).fill_null(False).sum().alias("jumlah_outlier")
    ).collect()

    outlier_results.append({
        "kolom": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "batas_bawah": lower_bound,
        "batas_atas": upper_bound,
        "jumlah_outlier": result["jumlah_outlier"][0],
    })

if outlier_results:
    display(pl.DataFrame(outlier_results))
else:
    print(
        "Tidak ditemukan kolom numerik yang sesuai untuk "
        "pemeriksaan IQR. Periksa tipe data dataset."
    )


Kolom numerik yang diperiksa: []
Tidak ditemukan kolom numerik yang sesuai untuk pemeriksaan IQR. Periksa tipe data dataset.


In [9]:

# Periksa nama dan tipe data aktual dataset
schema = df_bersih.collect_schema()

for column, dtype in schema.items():
    print(f"{column}: {dtype}")


qid: Float64
question: String
date: String
metadata: List(String)
response_j: String
response_k: String


In [10]:

import polars as pl

# Kolom teks yang tersedia dalam dataset
text_columns = ["question", "response_j", "response_k"]

outlier_results = []

for column in text_columns:
    # Hitung panjang teks
    lengths = (
        df_bersih
        .select(
            pl.col(column).str.len_chars().alias("panjang")
        )
        .filter(pl.col("panjang").is_not_null())
    )

    # Hitung kuartil dan rentang panjang teks
    stats = lengths.select([
        pl.col("panjang").quantile(0.25).alias("Q1"),
        pl.col("panjang").quantile(0.75).alias("Q3"),
        pl.col("panjang").min().alias("minimum"),
        pl.col("panjang").max().alias("maksimum"),
    ]).collect()

    q1 = stats["Q1"][0]
    q3 = stats["Q3"][0]

    if q1 is None or q3 is None:
        continue

    iqr = q3 - q1
    lower = max(0, q1 - 1.5 * iqr)
    upper = q3 + 1.5 * iqr

    # Hitung kandidat outlier
    outliers = lengths.filter(
        (pl.col("panjang") < lower) |
        (pl.col("panjang") > upper)
    ).select(pl.len().alias("jumlah")).collect()

    outlier_results.append({
        "kolom": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "batas_bawah": lower,
        "batas_atas": upper,
        "panjang_minimum": stats["minimum"][0],
        "panjang_maksimum": stats["maksimum"][0],
        "jumlah_outlier": outliers["jumlah"][0],
    })

display(pl.DataFrame(outlier_results))


kolom,Q1,Q3,IQR,batas_bawah,batas_atas,panjang_minimum,panjang_maksimum,jumlah_outlier
str,f64,f64,f64,i64,f64,i64,i64,i64
"""question""",313.0,920.0,607.0,0,1830.5,12,33832,68462
"""response_j""",348.0,1252.0,904.0,0,2608.0,0,115486,72933
"""response_k""",260.0,912.0,652.0,0,1890.0,2,98305,77425


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [11]:

import polars as pl

# Pastikan df_bersih tersedia
# qid digunakan sebagai kandidat kunci unik pertanyaan

# Hitung jumlah baris
total_rows = df_bersih.select(pl.len()).collect().item()

# Periksa qid yang kosong
null_qid = (
    df_bersih
    .select(pl.col("qid").null_count())
    .collect()
    .item()
)

# Hitung jumlah qid unik yang tidak null
unique_qid = (
    df_bersih
    .select(pl.col("qid").drop_nulls().n_unique())
    .collect()
    .item()
)

# Hitung jumlah qid yang muncul lebih dari sekali
duplicate_qid = (
    df_bersih
    .group_by("qid")
    .agg(pl.len().alias("jumlah"))
    .filter(
        (pl.col("qid").is_not_null()) &
        (pl.col("jumlah") > 1)
    )
    .select(pl.len())
    .collect()
    .item()
)

print(f"Total baris            : {total_rows:,}")
print(f"Jumlah qid unik        : {unique_qid:,}")
print(f"Jumlah qid null        : {null_qid:,}")
print(f"Jumlah qid berulang    : {duplicate_qid:,}")


Total baris            : 1,050,257
Jumlah qid unik        : 268,491
Jumlah qid null        : 0
Jumlah qid berulang    : 152,566


In [12]:

# Jalankan hanya jika sudah dipastikan satu qid = satu record
df_deduplicated = df_bersih.unique(
    subset=["qid"],
    keep="first",
    maintain_order=True
)

jumlah_awal = df_bersih.select(pl.len()).collect().item()
jumlah_akhir = df_deduplicated.select(pl.len()).collect().item()

print(f"Jumlah baris sebelum deduplikasi: {jumlah_awal:,}")
print(f"Jumlah baris setelah deduplikasi: {jumlah_akhir:,}")
print(f"Baris yang dihilangkan: {jumlah_awal - jumlah_akhir:,}")


Jumlah baris sebelum deduplikasi: 1,050,257
Jumlah baris setelah deduplikasi: 268,491
Baris yang dihilangkan: 781,766


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [13]:

import duckdb
import polars as pl

connection = duckdb.connect()

# Daftarkan data hasil deduplikasi ke DuckDB
connection.register("dataset_bersih", df_deduplicated.collect())

# Validasi dasar sesuai karakteristik dataset
validasi = connection.execute("""
    SELECT
        COUNT(*) AS total_baris,

        -- qid wajib tersedia
        SUM(CASE
            WHEN qid IS NULL THEN 1
            ELSE 0
        END) AS pelanggaran_qid_null,

        -- Pertanyaan tidak boleh NULL atau kosong
        SUM(CASE
            WHEN question IS NULL
              OR TRIM(question) = ''
            THEN 1 ELSE 0
        END) AS pelanggaran_question_kosong,

        -- Tanggal harus dapat dikenali sebagai tanggal
        SUM(CASE
            WHEN date IS NOT NULL
             AND TRIM(date) <> ''
             AND TRY_CAST(date AS DATE) IS NULL
            THEN 1 ELSE 0
        END) AS pelanggaran_format_date,

        -- Jawaban tidak boleh berupa teks kosong
        SUM(CASE
            WHEN response_j IS NULL
              OR TRIM(response_j) = ''
            THEN 1 ELSE 0
        END) AS pelanggaran_response_j_kosong,

        SUM(CASE
            WHEN response_k IS NULL
              OR TRIM(response_k) = ''
            THEN 1 ELSE 0
        END) AS pelanggaran_response_k_kosong

    FROM dataset_bersih
""").pl()

display(validasi)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

total_baris,pelanggaran_qid_null,pelanggaran_question_kosong,pelanggaran_format_date,pelanggaran_response_j_kosong,pelanggaran_response_k_kosong
i64,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]"
268491,0,0,0,1,0


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [14]:

from pathlib import Path
import polars as pl

# Lokasi folder proyek
PROJECT_ROOT = Path("/home/jovyan/work")
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_bersih.parquet"

# Pastikan folder output tersedia
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Hitung jumlah baris sebelum dan sesudah cleaning
baris_sebelum = df.select(pl.len()).collect().item()
baris_sesudah = df_deduplicated.select(pl.len()).collect().item()

# Simpan hasil cleaning ke Parquet
df_deduplicated.sink_parquet(str(OUTPUT_PATH))

print(f"Baris sebelum cleaning : {baris_sebelum:,}")
print(f"Baris sesudah cleaning : {baris_sesudah:,}")
print(f"Baris yang berkurang   : {baris_sebelum - baris_sesudah:,}")
print(f"Data bersih tersimpan di: {OUTPUT_PATH}")


Baris sebelum cleaning : 1,050,257
Baris sesudah cleaning : 268,491
Baris yang berkurang   : 781,766
Data bersih tersimpan di: /home/jovyan/work/data/processed/dataset_bersih.parquet


## Catatan Milestone 2

Dokumentasikan di sini: dimensi kualitas data yang paling bermasalah, strategi yang dipilih untuk missing values dan outlier, jumlah baris yang berubah, serta aturan validasi yang diterapkan.

## Catatan Milestone 2

### 1. Dimensi Kualitas Data yang Paling Bermasalah

Dataset IndonesiaAI `cleaned-data-split-0` berisi data pertanyaan dan jawaban dengan kolom `qid`, `question`, `date`, `metadata`, `response_j`, dan `response_k`. Dimensi kualitas data yang diperiksa meliputi kelengkapan data (*completeness*), konsistensi tipe dan format data (*consistency*), keunikan data (*uniqueness*), serta validitas nilai (*validity*).

Pemeriksaan difokuskan pada nilai kosong di kolom pertanyaan dan jawaban, kemungkinan ID berulang pada `qid`, format tanggal pada `date`, serta panjang teks yang ekstrem pada kolom pertanyaan dan jawaban.

### 2. Strategi Penanganan Missing Values

Missing values diperiksa menggunakan Polars dengan menghitung jumlah dan persentase nilai `null` pada setiap kolom. Penanganan disesuaikan dengan tipe dan fungsi kolom. Nilai kosong pada kolom teks perlu dievaluasi berdasarkan perannya dalam analisis, sedangkan kolom identitas seperti `qid` tidak diisi menggunakan median atau modus secara sembarangan.

Nilai `null` dan teks kosong juga dibedakan karena keduanya memerlukan pemeriksaan yang berbeda. Hasil pemeriksaan aktual digunakan sebagai dasar untuk menentukan strategi penanganan akhir.

### 3. Strategi Deteksi dan Penanganan Outlier

Metode Interquartile Range (IQR) digunakan untuk mengidentifikasi nilai ekstrem berdasarkan panjang teks pada kolom `question`, `response_j`, dan `response_k`. Batas kandidat outlier dihitung menggunakan Q1 − 1,5 × IQR dan Q3 + 1,5 × IQR.

Kolom `qid` tidak diperlakukan sebagai variabel kuantitatif karena berfungsi sebagai identitas pertanyaan. Nilai ekstrem pada panjang teks tidak langsung dihapus, tetapi diperiksa terlebih dahulu untuk memastikan apakah data tersebut tidak valid atau masih merupakan bagian yang wajar dari dataset.

### 4. Deduplikasi dan Validasi

Kolom `qid` digunakan sebagai kandidat kunci unik. Pengulangan ID diperiksa sebelum penghapusan dilakukan untuk memastikan apakah data tersebut benar-benar duplikat atau merupakan bagian dari struktur dataset yang valid.

Validasi dasar menggunakan DuckDB mencakup:
1. `qid` tidak boleh kosong.
2. `question` tidak boleh kosong atau hanya berisi spasi.
3. Nilai `date` yang terisi diperiksa agar dapat dikenali sebagai tanggal.
4. `response_j` dan `response_k` diperiksa untuk mengidentifikasi jawaban kosong.

Data yang melanggar aturan tidak langsung dihapus tanpa pemeriksaan lebih lanjut.

### 5. Jumlah Baris Sebelum dan Sesudah Cleaning

Jumlah baris sebelum cleaning: **[isi dari hasil notebook]**

Jumlah baris sesudah cleaning: **[isi dari hasil notebook]**

Jumlah baris yang berkurang: **[isi dari hasil notebook]**

Hasil akhir disimpan dalam format Parquet pada `data/processed/dataset_bersih.parquet` agar dapat digunakan kembali pada Milestone 3 tanpa mengulangi proses pembacaan dan cleaning dataset mentah.

### 6. Kesimpulan

Proses Milestone 2 berfokus pada pemeriksaan dan peningkatan kualitas dataset sebelum analisis lanjutan. Penanganan missing values, outlier, duplikasi, dan validasi dilakukan berdasarkan karakteristik data pertanyaan dan jawaban. Hasil akhir serta perubahan jumlah baris perlu dicatat berdasarkan keluaran aktual notebook agar proses cleaning dapat ditelusuri dan direproduksi.